In [2]:
# Autogluon ile regresyon modeli eğitimi için gerekli dönüşümleri yapalım
from autogluon.tabular import TabularPredictor
import pandas as pd
import numpy as np
import os

In [4]:
# Veriyi yükle
df = pd.read_csv("all_data_ratio.csv")

In [6]:
# İlgili hedefler ve feature map
hedefler = ["uzman", "pratisyen", "asistan", "ebe", "diger", "hemsire", "yatak", "hastane"]
feature_map = {
    "uzman": ["yıl", "nufus", "hastane", "yatak", "asistan", "pratisyen", "hemsire", "ebe", "diger"],
    "pratisyen": ["yıl", "nufus", "hastane", "yatak", "asistan", "uzman", "hemsire", "ebe", "diger"],
    "asistan": ["yıl", "nufus", "hastane", "yatak", "uzman", "pratisyen", "hemsire", "ebe", "diger"],
    "ebe": ["yıl", "nufus", "hastane", "yatak", "asistan", "uzman", "hemsire", "pratisyen", "diger"],
    "diger": ["yıl", "nufus", "hastane", "yatak", "asistan", "pratisyen", "hemsire", "ebe", "uzman"],
    "hemsire": ["yıl", "nufus", "hastane", "yatak", "asistan", "uzman", "pratisyen", "ebe", "diger"],
    "yatak": ["yıl", "nufus", "hastane", "hemsire", "asistan", "uzman", "pratisyen", "ebe", "diger"],
    "hastane": ["yıl", "nufus", "yatak", "hemsire", "asistan", "uzman", "pratisyen", "ebe", "diger"]
}

In [ ]:
# Hedefe özel eğitim süreleri
hedef_time_map = {
    "uzman": 180,
    "hemsire": 180,
    "pratisyen": 120,
    "asistan": 120,
    "ebe": 60,
    "diger": 60,
    "yatak": 90,
    "hastane": 90
}

autogluon_tahminler = []

for il in df["il_adı"].unique():
    df_il = df[df["il_adı"] == il]

    for hedef in hedefler:
        if df_il[df_il["yıl"] <= 2021].shape[0] < 5 or hedef not in df_il.columns:
            continue

        features = feature_map[hedef]
        gerekli_kolonlar = features + [hedef]
        if any(f not in df_il.columns for f in gerekli_kolonlar):
            continue

        df_target = df_il[gerekli_kolonlar].dropna()
        if df_target.empty or df_target[df_target["yıl"] <= 2021].shape[0] < 4:
            continue

        train = df_target[df_target["yıl"] <= 2021]
        val = df_target[df_target["yıl"] == 2022]
        test = df_target[df_target["yıl"] == 2023]

        if val.empty or test.empty or val[features].shape[0] == 0 or test[features].shape[0] == 0:
            continue

        try:
            # Hedefe göre süre seçimi
            time_limit = hedef_time_map.get(hedef, 60)

            # Model eğitimi
            predictor = TabularPredictor(label=hedef, path=f"/mnt/data/ag_{il}_{hedef}_model").fit(
                train_data=train[features + [hedef]],
                time_limit=time_limit,
                presets="best_quality"
            )

            # Tahminler
            y_val_pred = predictor.predict(val[features]).values[0]
            y_test_pred = predictor.predict(test[features]).values[0]

            autogluon_tahminler.append({
                "il_adı": il,
                "hedef": hedef,
                "gercek_2022": val[hedef].values[0],
                "tahmin_2022_ag": y_val_pred,
                "gercek_2023": test[hedef].values[0],
                "tahmin_2023_ag": y_test_pred,
            })

        except Exception as e:
            print(f"{il} - {hedef} modeli eğitilemedi. Hata: {e}")
            continue

# Tahminleri DataFrame'e çevir
df_ag = pd.DataFrame(autogluon_tahminler)

# Doğruluk yüzdesi hesapla
df_ag["yuzde_dog"] = np.where(
    df_ag["gercek_2023"] == 0,
    0,
    100 - np.abs(df_ag["gercek_2023"] - df_ag["tahmin_2023_ag"]) / df_ag["gercek_2023"] * 100
)
df_ag["yuzde_dog"] = df_ag["yuzde_dog"].clip(0, 100)

Verbosity: 2 (Standard Logging)
=================== System Info ===================
AutoGluon Version:  1.3.0
Python Version:     3.12.7
Operating System:   Windows
Platform Machine:   AMD64
Platform Version:   10.0.22631
CPU Count:          22
Memory Avail:       4.00 GB / 15.61 GB (25.6%)
Disk Space Avail:   787.15 GB / 951.65 GB (82.7%)
Presets specified: ['best_quality']
Setting dynamic_stacking from 'auto' to True. Reason: Enable dynamic_stacking when use_bag_holdout is disabled. (use_bag_holdout=False)
Stack configuration (auto_stack=True): num_stack_levels=1, num_bag_folds=5, num_bag_sets=1
DyStack is enabled (dynamic_stacking=True). AutoGluon will try to determine whether the input data is affected by stacked overfitting and enable or disable stacking as a consequence.
	This is used to identify the optimal `num_stack_levels` value. Copies of AutoGluon will be fit on subsets of the data. Then holdout validation data is used to detect stacked overfitting.
	Running DyStack for up 

In [ ]:
from sklearn.metrics import mean_absolute_error, mean_squared_error

y_true = df_ag["gercek_2023"]
y_pred = df_ag["tahmin_2023_ag"]
mae = mean_absolute_error(y_true, y_pred)
rmse = mean_squared_error(y_true, y_pred, squared=False)
mape = np.mean(np.abs((y_true - y_pred) / y_true.replace(0, np.nan))) * 100  # 0 bölmeleri hariç

# Kaydet
df_ag.to_csv("sehir_bazli_2023_autogluon_tahmin.csv", index=False)

# Sonuçları göster
import ace_tools as tools; tools.display_dataframe_to_user(name="AutoGluon Tahmin Sonuçları", dataframe=df_ag)

(mae, rmse, mape)

In [ ]:
# Türkiye'nin 7 coğrafi bölgesi
bolge_haritasi = {
    'Marmara': ["İstanbul", "Edirne", "Kırklareli", "Tekirdağ", "Kocaeli", "Sakarya", "Yalova", "Bilecik", "Balıkesir", "Çanakkale", "Bursa"],
    'Ege': ["İzmir", "Aydın", "Muğla", "Manisa", "Afyonkarahisar", "Uşak", "Kütahya", "Denizli"],
    'Akdeniz': ["Antalya", "Mersin", "Adana", "Hatay", "Kahramanmaraş", "Osmaniye", "Isparta", "Burdur"],
    'İç Anadolu': ["Ankara", "Konya", "Kayseri", "Eskişehir", "Nevşehir", "Aksaray", "Kırıkkale", "Niğde", "Karaman", "Sivas", "Yozgat", "Kırşehir", "Çankırı"],
    'Karadeniz': ["Samsun", "Trabzon", "Ordu", "Giresun", "Rize", "Artvin", "Tokat", "Amasya", "Bartın", "Zonguldak", "Karabük", "Sinop", "Çorum", "Bayburt", "Gümüşhane", "Kastamonu", "Bolu", "Düzce"],
    'Doğu Anadolu': ["Erzurum", "Van", "Malatya", "Elazığ", "Kars", "Ağrı", "Bitlis", "Iğdır", "Tunceli", "Ardahan", "Erzincan", "Bingöl", "Muş", "Hakkari"],
    'Güneydoğu Anadolu': ["Diyarbakır", "Gaziantep", "Şanlıurfa", "Mardin", "Batman", "Siirt", "Şırnak", "Adıyaman", "Kilis"]
}

def bolge_bul(il):
    for bolge, iller in bolge_haritasi.items():
        if il in iller:
            return bolge
    return "Bilinmiyor"

# df_ag = AutoGluon şehir bazlı tahminlerin bulunduğu DataFrame
df_ag["bolge"] = df_ag["il_adı"].apply(bolge_bul)

# Bölge bazında toplam gerçek ve tahmin değerlerini hesapla
bolgesel_ag = df_ag.groupby(["bolge", "hedef"])[["gercek_2023", "tahmin_2023_ag"]].sum().reset_index()

# İstersen CSV olarak da kaydedebilirsin
bolgesel_ag.to_csv("bolge_bazli_2023_autogluon_tahmin.csv", index=False)


In [ ]:
# 1. Dosyayı oku
df = pd.read_csv("sehir_bazli_2023_autogluon_tahmin.csv")  # Dosya adını gerektiğinde değiştir

# 2. AutoGluon için doğruluk hesapla
df["ag_yuzde_dog"] = np.where(
    df["gercek_2023"] == 0,
    0,
    100 - (np.abs(df["gercek_2023"] - df["tahmin_2023_ag"]) / df["gercek_2023"] * 100)
)
df["ag_yuzde_dog"] = df["ag_yuzde_dog"].clip(0, 100)

# 3. Ortalama doğruluğu yazdır
ortalama = round(df["ag_yuzde_dog"].mean(), 2)
print(f"\nAutoGluon Şehir Bazlı Ortalama Doğruluk (2023): {ortalama}%")

# 4. Güncellenmiş dosyayı kaydet
df.to_csv("sehir_autogluon_yuzdeler.csv", index=False)

In [ ]:
# 1. Dosyayı oku
df = pd.read_csv("bolge_bazli_2023_autogluon_tahmin.csv")  # Dosya adını gerektiğinde değiştir

# 2. Doğruluk hesapla
df["ag_yuzde_dog"] = np.where(
    df["gercek_2023"] == 0,
    0,
    100 - (np.abs(df["gercek_2023"] - df["tahmin_2023_ag"]) / df["gercek_2023"] * 100)
)
df["ag_yuzde_dog"] = df["ag_yuzde_dog"].clip(0, 100)

# 3. Ortalama doğruluğu yazdır
ortalama = round(df["ag_yuzde_dog"].mean(), 2)
print(f"\nAutoGluon Bölge Bazlı Ortalama Doğruluk (2023): {ortalama}%")

# 4. Güncellenmiş dosyayı kaydet
df.to_csv("bolge_autogluon_yuzdeler.csv", index=False)